# Smart Recycling Assistant Model Training

This notebook trains a YOLO object detection model to detect:

1. Glass
2. Metal
3. Paper
4. Plastic
5. Waste

### Install the packages

In [1]:
%pip install -U ultralytics

Note: you may need to restart the kernel to use updated packages.


After this finishes, restart the notebook kernel if VS Code asks you to.

### Import libraries

In [2]:
from pathlib import Path

import torch
from ultralytics import YOLO

### Check which GPU is available

In [3]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = 0
elif torch.backends.mps.is_available():
    print("Apple Silicon GPU available")
    DEVICE = "mps"
else:
    print("No supported GPU detected. Training will use the CPU.")
    DEVICE = "cpu"

print("Selected device:", DEVICE)

PyTorch version: 2.13.0
CUDA available: False
Apple Silicon GPU available
Selected device: mps


### Set the project paths

Works whether this notebook sits in the repo root or in a `notebooks/` subfolder.

In [4]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent  # step up so DATA_DIR is always correct

DATA_DIR = PROJECT_ROOT / "data"
DATA_YAML = DATA_DIR / "data.yaml"

print("Project root:", PROJECT_ROOT)
print("Dataset YAML:", DATA_YAML)
print("YAML exists:", DATA_YAML.exists())

Project root: /Users/hafsahnasir/Developer/SNU/project/eco-vision
Dataset YAML: /Users/hafsahnasir/Developer/SNU/project/eco-vision/data/data.yaml
YAML exists: True


### Read and display `data.yaml`

In [5]:
print(DATA_YAML.read_text())

# No `path:` key on purpose — Ultralytics then resolves these relative to THIS
# file's own folder (data/), so it works on macOS/Windows from any working dir.
train: train/images
val: valid/images
test: test/images

nc: 5

names:
  0: Glass
  1: Metal
  2: Paper
  3: Plastic
  4: Waste



### Check the dataset folders

In [6]:
splits = ["train", "valid", "test"]

for split in splits:
    image_dir = DATA_DIR / split / "images"
    label_dir = DATA_DIR / split / "labels"

    image_files = (
        list(image_dir.glob("*.jpg"))
        + list(image_dir.glob("*.jpeg"))
        + list(image_dir.glob("*.png"))
    )
    label_files = list(label_dir.glob("*.txt"))

    print(f"\n{split.upper()}")
    print("Images:", len(image_files))
    print("Labels:", len(label_files))


TRAIN
Images: 3502
Labels: 3502

VALID
Images: 580
Labels: 580

TEST
Images: 45
Labels: 45


### Load a pretrained model

Use a small model first so training is faster. `n` is the smallest (nano) variant.

In [10]:
model = YOLO("yolo11n.pt")

### Train the model

In [11]:
training_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=8,
    device=DEVICE,
    project=str(PROJECT_ROOT / "runs"),
    name="recycling_baseline",
    exist_ok=True,
    patience=15,
    pretrained=True,
    plots=True,
)

Ultralytics 8.4.95 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M2 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/hafsahnasir/Developer/SNU/project/eco-vision/data/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=recycling_baseline, nbs=64, nms=False, opset=None, optimize=Fal

/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50       2.3G      1.296      3.147      1.218         12        640: 100% ━━━━━━━━━━━━ 438/438 2.5it/s 2:580.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.4it/s 15.3s0.4s
                   all        580       1319      0.329      0.146     0.0816     0.0501

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      2.19G      1.376      2.883      1.233         26        640: 0% ──────────── 0/438  1.1s

/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      3.26G      1.434        2.7      1.284         18        640: 100% ━━━━━━━━━━━━ 438/438 2.6it/s 2:510.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.3it/s 15.8s0.4s
                   all        580       1319      0.347      0.181        0.1      0.061

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       3/50      3.25G      1.424      2.457      1.275         14        640: 100% ━━━━━━━━━━━━ 438/438 2.6it/s 2:490.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.2it/s 16.6s0.4s
                   all        580       1319      0.361      0.199      0.101     0.0637

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      2.26G      1.705      2.808      1.164         35        640: 0% ──────────── 0/438  1.1s

/Users/hafsahnasir/Developer/SNU/project/eco-vision/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       4/50      3.25G      1.435      2.363      1.282         34        640: 100% ━━━━━━━━━━━━ 438/438 2.5it/s 2:580.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 75% ━━━━━━━━━─── 28/37 1.5it/s 13.2s<6.1s


KeyboardInterrupt: 

**What these settings mean**

- `epochs=50`: the model goes through the training data up to 50 times.
- `imgsz=640`: images are resized to 640 x 640.
- `batch=8`: eight images are processed together.
- `patience=15`: training stops early if validation performance does not improve for 15 epochs.
- `plots=True`: saves graphs and a confusion matrix.
- `pretrained=True`: starts from pretrained weights instead of training from zero.

If you get an out-of-memory error, reduce `batch=4`. If training is too slow on your Mac, start smaller: `epochs=20, imgsz=416, batch=4`.

### Show where the results were saved

In [ ]:
RUN_DIR = PROJECT_ROOT / "runs" / "recycling_baseline"

print("Training output:", RUN_DIR)
print("Best model:", RUN_DIR / "weights" / "best.pt")
print("Last model:", RUN_DIR / "weights" / "last.pt")

### Load the best model

In [ ]:
BEST_MODEL_PATH = RUN_DIR / "weights" / "best.pt"

assert BEST_MODEL_PATH.exists(), "The best model file was not found."

best_model = YOLO(str(BEST_MODEL_PATH))

print("Loaded:", BEST_MODEL_PATH)

### Evaluate it on the validation set

In [ ]:
validation_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    device=DEVICE,
    plots=True,
)

print("mAP50:", validation_metrics.box.map50)
print("mAP50-95:", validation_metrics.box.map)
print("Precision:", validation_metrics.box.mp)
print("Recall:", validation_metrics.box.mr)

### Evaluate it on the test set

In [ ]:
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    device=DEVICE,
    plots=True,
)

print("Test mAP50:", test_metrics.box.map50)
print("Test mAP50-95:", test_metrics.box.map)
print("Test precision:", test_metrics.box.mp)
print("Test recall:", test_metrics.box.mr)

### Run predictions on test images

In [ ]:
TEST_IMAGES = DATA_DIR / "test" / "images"

prediction_results = best_model.predict(
    source=str(TEST_IMAGES),
    conf=0.50,
    imgsz=640,
    device=DEVICE,
    save=True,
    project=str(PROJECT_ROOT / "runs"),
    name="test_predictions",
    exist_ok=True,
)

### Display a prediction inside the notebook

In [ ]:
import matplotlib.pyplot as plt

if prediction_results:
    predicted_image = prediction_results[0].plot()

    plt.figure(figsize=(10, 8))
    plt.imshow(predicted_image[..., ::-1])
    plt.axis("off")
    plt.show()

### Copy the final model into a `models` folder

In [ ]:
import shutil

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

FINAL_MODEL_PATH = MODELS_DIR / "recycling_model.pt"

shutil.copy2(BEST_MODEL_PATH, FINAL_MODEL_PATH)

print("Final model copied to:", FINAL_MODEL_PATH)